# Fine-tune your first LLM, then serve it

We are going to take a small open model and teach it one specific job: **turn a question about a
database into a SQL query**. Then we save it in a form that a real inference server can load.

Out of the box the model answers that kind of question with a paragraph of explanation and a code
block. After ~10 minutes of training it will answer with one line of SQL and nothing else. That
difference is the whole point of fine-tuning: you are not teaching the model new facts, you are
teaching it *how to respond*.

**Two words you will see everywhere:**

- **LoRA** — instead of updating all 1.5 billion weights, we freeze them and train a few small
  extra matrices bolted onto the model. ~1% as many numbers to learn, and the result is a file of
  a few dozen MB called an *adapter*.
- **QLoRA** — the same thing, but the frozen original is also squashed down to 4 bits per weight so
  it takes a quarter of the memory. That is what makes this fit on a free Kaggle GPU.

**Before you run anything:** in the panel on the right, set **Accelerator → GPU T4 x2**.

## 1. Set up

Kaggle already has most of this, but the libraries move fast and the training API changed
recently, so we upgrade them. This takes a couple of minutes.

In [ ]:
%pip install -q -U transformers trl peft bitsandbytes datasets accelerate

In [ ]:
!nvidia-smi

Two **Tesla T4** cards, 15 GB each.

One thing to note now, because it explains a setting later: the T4 is an older GPU. It cannot do
`bfloat16`, the number format most modern fine-tuning tutorials use. It can do `float16`, so that
is what we use everywhere below. On a newer card (A100, H100) you would switch those to `bfloat16`
and nothing else would change.

In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

# What we want the model to be. It goes at the top of every training example, and
# it has to be there at inference time too -- see the `ask` function further down.
SYSTEM = "You are a SQL generator. Given a table schema and a question, reply with one SQL query and nothing else."

## 2. The dataset

`b-mc2/sql-create-context` is ~78,000 examples of exactly the job we want. We take the first 2000,
which is plenty to change the model's behaviour.

Look at the row printed below: three flat text columns, no chat formatting anywhere. Almost every
dataset you find will look like this. Turning it into a conversation is our job, and it is the
next step.

In [ ]:
data = load_dataset("b-mc2/sql-create-context", split="train[:2000]")

print(data)
print()
for key, value in data[0].items():
    print(f"{key}: {value}")

## 3. Chat templates — the step people skip

An instruct model does not read plain text. It was trained on a conversation written in a very
specific format, with special marker tokens around each turn, and it only behaves properly if you
hand it that exact format. Every model family has its own — Qwen's markers are `<|im_start|>` and
`<|im_end|>`, Llama's look completely different.

You never write those markers yourself. You write a list of messages with roles, and
`tokenizer.apply_chat_template` renders it into whatever this particular model expects. The recipe
is baked into the tokenizer, so switching models switches the format for free.

Three roles here:

| role | what goes in it |
|---|---|
| `system` | the standing instruction — our `SYSTEM` string |
| `user` | the schema plus the question |
| `assistant` | the answer we want, i.e. what the model is learning to produce |

Run the cell and read the rendered string. That, verbatim, is what the model trains on.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def to_text(row):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"{row['context']}\n\n{row['question']}"},
        {"role": "assistant", "content": row["answer"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

print(to_text(data[0])["text"])

Now apply it to all 2000 rows. We drop the original columns and keep only `text` — one rendered
conversation per row is all the trainer needs.

In [ ]:
data = data.map(to_text, remove_columns=data.column_names)

print(data)

## 4. Deciding the quantization

Every weight in the model is a number, and you choose how many bits to spend on each one. Fewer
bits means a smaller model and less GPU memory, at some cost in quality.

For our 1.5B model:

| precision | size | notes |
|---|---|---|
| `float32` | ~6 GB | full precision, no reason to use it here |
| `float16` | ~3 GB | the normal choice for inference |
| 8-bit | ~1.6 GB | quality loss is hard to notice |
| **4-bit (NF4)** | **~1 GB** | what we use — the frozen base does not need to be precise |

4-bit is safe here specifically *because* of LoRA: the squashed weights are frozen and never
updated, and the parts that actually learn (the adapter) stay in full precision.

Two extra options in the config below:

- `nf4` — a 4-bit format shaped for how neural network weights are actually distributed, better
  than plain 4-bit integers.
- `double_quant` — quantizes the quantization constants too. Saves another ~0.4 bits per weight
  for free.

A 1.5B model at 4-bit would fit on one T4 with room to spare — so we cap GPU 0's budget at 1 GB to
force the rest of the layers onto GPU 1, and use both cards. Print `hf_device_map` after loading to
see exactly which layer landed where.

In [ ]:
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,   # bfloat16 on a newer GPU
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=quantization,
    device_map="auto",
    max_memory={0: "1GiB", 1: "14GiB"},     # raise GPU 0's share if you hit an out-of-memory error
)
model = prepare_model_for_kbit_training(model)

print(model.hf_device_map)

### Two GPUs, two different ways

What we just did is **model parallelism**: the model is cut in half, first layers on GPU 0, rest on
GPU 1. Data flows through card 0, then card 1. It uses both cards, and it is the only reason very
large models run at all — but it is not faster, because only one card is busy at a time.

The other way is **data parallelism**: a full copy of the model on each GPU, each copy training on
a different slice of the data, gradients averaged after every step. That one really is ~2x faster,
but it needs one process per GPU, which a notebook cannot do. It is what
`hpc/training/llm-FineTuning.py` does with `torchrun`, across 2 nodes x 2 GPUs.

Same recipe, different plumbing. Learn it here, scale it there.

## 5. Ask the model *before* training

This is the half of the before/after that tutorials always forget. Same question we will ask again
at the end.

Note `add_generation_prompt=True` — at training time the assistant's answer was in the text; now we
want the template to stop right where the answer *starts*, so the model fills it in.

In [ ]:
SCHEMA = "CREATE TABLE head (age INTEGER)"
QUESTION = "How many heads of the departments are older than 56?"

def ask(model, schema, question):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"{schema}\n\n{question}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    print(tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

ask(model, SCHEMA, QUESTION)

It knows the answer. It just will not shut up about it — an explanation, a markdown code fence,
probably an offer to help further. If you piped that into a database it would fail.

That is the behaviour we are about to change.

## 6. The LoRA adapter

`target_modules` is the list of layers we attach adapters to: the four attention projections and
the three MLP ones. That covers everything worth training in a transformer block.

- `r=16` — the size of the adapter matrices. Bigger learns more and overfits sooner; 8–64 is the
  usual range.
- `lora_alpha=32` — how loudly the adapter speaks relative to the frozen model. Convention is 2x `r`.
- `lora_dropout=0.05` — mild regularisation.

The number printed at the end is the point of all this.

In [ ]:
lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

## 7. Train

`SFT` is *supervised fine-tuning*: show the model finished examples, have it predict them one token
at a time, nudge the adapter whenever it is wrong. `SFTTrainer` handles the tokenizing, batching and
the training loop.

The settings that matter:

- `max_steps=60` — we stop after 60 steps rather than a full pass, so this finishes in minutes. Drop
  it (and keep `num_train_epochs`) for a real run.
- batch size 2 x 4 accumulation = an **effective batch of 8**. Accumulation is how you get a big
  batch on a small card: run 4 small batches, add the gradients up, then take one step.
- `fp16=True` — the T4 rule from the top of the notebook.
- `gradient_checkpointing=True` — throws away intermediate activations and recomputes them in the
  backward pass. Slower, much less memory. Standard for fine-tuning.
- `learning_rate=2e-4` — high compared to full fine-tuning, normal for LoRA. Only the adapter moves.

Watch the loss column. It should fall, unsteadily, and flatten out.

In [ ]:
config = SFTConfig(
    output_dir="/kaggle/working/out",
    dataset_text_field="text",
    max_length=1024,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_steps=60,                  # remove for a full run
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,
    save_strategy="no",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=config,
    train_dataset=data,
    peft_config=lora,
    processing_class=tokenizer,
)

trainer.model.print_trainable_parameters()

In [ ]:
trainer.train()

## 8. Ask the exact same question again

In [ ]:
trainer.model.config.use_cache = True   # was off during training, makes generation fast again

ask(trainer.model, SCHEMA, QUESTION)

One line of SQL. No preamble, no code fence, no follow-up offer.

Nothing about the model's knowledge changed in those 60 steps — it could already write this query.
What changed is its idea of what a good answer looks like. That is what supervised fine-tuning
does, and it is worth being clear-eyed about it: if you want the model to know something it does
not know, fine-tuning is usually the wrong tool, and retrieval is the right one.

## 9. Save it

Two different things you can save, and the difference matters when you go to serve it:

- **The adapter** — tens of MB. Just the trained matrices. Useless on its own; whoever loads it
  needs the original Qwen model too. Great for keeping 20 variants around.
- **The merged model** — ~3 GB. Adapter arithmetic folded back into the weights, producing an
  ordinary standalone model. This is what vLLM, SGLang and llama.cpp want.

We save both. For the merge we reload the *original* model in fp16 on CPU rather than merging into
our 4-bit copy — merging into the squashed weights would bake the quantization error in permanently.

In [ ]:
trainer.model.save_pretrained("/kaggle/working/adapter")
tokenizer.save_pretrained("/kaggle/working/adapter")

del trainer, model
torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, device_map="cpu")
merged = PeftModel.from_pretrained(base, "/kaggle/working/adapter").merge_and_unload()
merged.save_pretrained("/kaggle/working/merged")
tokenizer.save_pretrained("/kaggle/working/merged")

!du -sh /kaggle/working/adapter /kaggle/working/merged

In [ ]:
# Optional: push it to the Hub so a server can pull it by name.
# Add your token under Add-ons -> Secrets, then:
#
# from huggingface_hub import login
# login(token=...)
# merged.push_to_hub("your-username/qwen2.5-1.5b-sql")
# tokenizer.push_to_hub("your-username/qwen2.5-1.5b-sql")

## 10. Now serve it

Download `/kaggle/working/merged` from the output panel on the right — that folder is a complete
model, and it is the input to every serving option in this repo:

| | |
|---|---|
| `hpc/serving/vllm/` | the usual choice. OpenAI-compatible API, fast, scales across GPUs |
| `hpc/serving/sglang/` | similar, faster on some workloads |
| `hpc/serving/llamacpp/` | needs a GGUF conversion first; runs happily on a CPU too |

Each of those folders has a `how-to-run.txt` (one GPU, copy-paste) and a `run.sh`
(multi-node, `sbatch`).

And to train this properly instead of for 60 steps, `hpc/training/` is this same notebook as a
script, running on 2 nodes x 2 GPUs. Start with `hpc/setup-env.txt`.